# P30 — Reflexion: agentes de lenguaje con refuerzo verbal

## 1. Título y paper

**Paper:** *Reflexion: Language Agents with Verbal Reinforcement Learning*  
**Autoría:** Noah Shinn, Federico Cassano, Ashwin Gopinath, Karthik Narasimhan, Shunyu Yao  
**Año y venue:** 2023 · arXiv:2303.11366 · NeurIPS 2023  
**Nivel:** L2 · **Motor:** `reflexion`  
**Ficha completa:** [`P30_reflexion`](../../papers/foundational/P30_reflexion/README.md)

**Hito:** El agente aprende entre intentos sin tocar un solo peso: el refuerzo ocurre en el contexto, en lenguaje natural.

- [arXiv:2303.11366](https://arxiv.org/abs/2303.11366)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un bucle ReAct que falla vuelve a empezar de cero y repite el mismo error, porque no conserva nada de lo aprendido en el intento anterior.
2. Ejecutar una implementación mínima de la propuesta: Tras cada fallo, generar una reflexión verbal sobre qué salió mal y conservarla en una memoria episódica que condiciona el siguiente intento.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P13
- P28


## 4. Intuición

Un agente sin memoria de sus fallos es alguien que repite el mismo error con entusiasmo. Reflexion añade lo mínimo para romper el bucle: escribir qué salió mal y leerlo antes de reintentar.


## 5. Concepto mínimo

```text
Bucle sin reflexión:   intento → falla → intento (idéntico) → falla → …

Bucle con reflexión:   intento → falla → REFLEXIÓN («olvidé el caso vacío»)
                       → memoria → intento (condicionado) → …
```

No hay gradientes. La política mejora porque **el contexto del siguiente intento es distinto**: es refuerzo, pero expresado en lenguaje.


## 6. Código explicado

El motor compara cuatro intentos con y sin memoria verbal del fallo.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('reflexion', seed=7)['result']
print('errores del problema:', r['errores_del_problema'], '\n')
print('SIN reflexión:')
for t in r['sin_reflexion']['traza']:
    print('  ', t)
print('\nCON reflexión:')
for t in r['con_reflexion']['traza']:
    print('  ', t)

## 7. Predicción antes de ejecutar

1. ¿Cuántos intentos necesita el agente sin memoria para superar tres errores distintos?
2. ¿Cuántos con memoria?
3. ¿Cuántos pesos se actualizan en el proceso?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('reflexion', seed=7)['result']
print('pesos actualizados:', r['pesos_actualizados'])
print('sin reflexión → éxito:', r['sin_reflexion']['exito'],
      '· intentos:', r['sin_reflexion']['intentos_usados'])
print('con reflexión → éxito:', r['con_reflexion']['exito'],
      '· intentos:', r['con_reflexion']['intentos_usados'])

## 9. Salida interpretable

Sin memoria, el agente no termina: repite el primer fallo indefinidamente. Con memoria, cada intento elimina un error y converge. Y **cero pesos actualizados**: todo el aprendizaje vive en el contexto, lo que lo hace barato, inmediato y también efímero.


## 10. Comentario pedagógico

El método depende por completo de tener una **señal de fallo fiable**: un test que falle, un compilador que proteste, un entorno que devuelva error. Sin verificador no hay sobre qué reflexionar, y la reflexión degenera en autoafirmación.


## 11. Error o anti-patrón deliberado

Anti-patrón: reflexionar sin señal externa, dejando que el modelo juzgue su propio trabajo sin evidencia.


In [ ]:
print('«Revisa tu respuesta y mejórala» sin ejecutar nada:')
print('  - el modelo suele declararse satisfecho, o cambia cosas al azar')
print('  - sin senal externa, la reflexion no tiene informacion nueva que incorporar')

## 12. Corrección

La corrección es anclar la reflexión en una observación verificable:


In [ ]:
ciclo = {
    '1_ejecutar': 'correr los tests / el código / la consulta',
    '2_observar': 'capturar el error concreto, no una impresión',
    '3_reflexionar': 'escribir qué causó ESE error',
    '4_reintentar': 'con la reflexión en el contexto',
    'criterio_de_parada': 'máximo de intentos + detección de reflexión repetida',
}
show(ciclo)

## 13. Desafío guiado

Comprueba qué pasa si la memoria crece sin límite: el contexto es finito.


In [ ]:
for intentos in (3, 10, 50, 200):
    tokens = intentos * 80
    print(f'{intentos:>3} intentos → ~{tokens:>6} tokens de memoria verbal '
          f"({'cabe' if tokens < 8000 else 'ya no cabe: hay que resumir o priorizar'})")

## 14. Desafío autónomo

Implementa Reflexion sobre un conjunto de ejercicios de programación con tests. Mide la tasa de éxito acumulada por número de intentos, con y sin reflexión, y cuenta cuántas reflexiones son realmente accionables frente a genéricas («ser más cuidadoso»).


## 15. Evidencia de aprendizaje

Guarda ambas trazas, el número de pesos actualizados y tu ciclo de reflexión anclado en observación verificable.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P30_reflexion/README.md) · evaluación formal: [`assessments/papers/P30_reflexion.md`](../../assessments/papers/P30_reflexion.md)


## 16. Cierre

El agente ya aprende de sus fallos dentro de una tarea. Falta que recuerde **entre** tareas y a lo largo del tiempo.


## 17. Conexión con el siguiente hito

- P16
- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
